# Análisis de Atrasos — Clientes con DPD > 15

Identifica las líneas de crédito PAY que han tenido al menos una cuota con más de 15 días de atraso y determina si siguen activas en la cartera.

In [ ]:
import pandas as pd
import numpy as np
from datetime import date
from dateutil.relativedelta import relativedelta

ruta_archivo = "C:/Users/AnaMaríaRamírezLizár/PROYECTOS/data/Base de datos.xlsx"

## 1. Carga de datos y reconstrucción de payc

In [ ]:
data_credito  = pd.read_excel(ruta_archivo, sheet_name='Creditos')
data_abonos   = pd.read_excel(ruta_archivo, sheet_name='Abonos')
data_clientes = pd.read_excel(ruta_archivo, sheet_name='Clientes')

# Personas morales - producto PAY
data_pm  = data_clientes.loc[data_clientes['Person Type'] == 'MORAL', 'Person Id']
dcred_pm = data_credito[data_credito['Person Id'].isin(data_pm)]

data_ldc = dcred_pm[(dcred_pm['Is line'] == 'SI') & (dcred_pm['Loan Status'] == 'AUTORIZADO')].copy()
data_ldc = data_ldc.rename(columns={'Cosecha': 'Originacion'})
data_ldc['RFC'] = data_ldc['Person Id'].map(data_clientes.set_index('Person Id')['Taxpayer ID Number'])
data_ldc['Status'] = np.where(
    pd.to_datetime(data_ldc['Due Date']) < pd.to_datetime(date.today()), 'INACTIVO', 'ACTIVO'
)

data_min = dcred_pm[dcred_pm['Loan Type'] == 'MINISTRACION'].copy()
data_min['Originacion'] = data_min['Line'].map(data_ldc.set_index('Line')['Originacion'])
data_min['RFC']         = data_min['Person Id'].map(data_clientes.set_index('Person Id')['Taxpayer ID Number'])

pay  = data_min[data_min['Product'].str.contains('PAY')].copy()
payb = data_abonos[data_abonos['Product'].str.contains('PAY')].copy()
pay  = pay.rename(columns={'Period Number': 'Cuotas'})
pay['Capital cuota'] = (pay['Amount'] / pay['Cuotas']).round(2)
payb = payb.rename(columns={'Installment': 'Cuota'})

pagos_agrup = (
    payb.groupby(['Portfolio Id', 'Cuota'])
    .agg({'Application Date': 'max', 'Capital': 'sum', 'Penalty': 'sum'})
    .reset_index()
)

pagos = pay.loc[pay.index.repeat(pay['Cuotas'])].assign(
    Cuota=lambda x: x.groupby('Portfolio Id').cumcount() + 1
)
payc = pagos.merge(pagos_agrup, on=['Portfolio Id', 'Cuota'], how='left')
payc['Application Date'] = pd.to_datetime(payc['Application Date']).fillna(date.today())
payc['Moratorios'] = payc['Penalty'].fillna(0)

def fcp(row):
    duedate = pd.to_datetime(row['Opening Date']) + relativedelta(months=row['Cuota'])
    return pd.to_datetime(duedate)

payc['Fecha vencimiento cuota'] = payc.apply(fcp, axis=1)
payc = payc[pd.to_datetime(payc['Fecha vencimiento cuota']) <= pd.to_datetime(date.today())]

def dpdp(row):
    DPDUP = pd.to_numeric((row['Application Date'] - row['Fecha vencimiento cuota']).days, errors='coerce')
    if DPDUP < 0:
        return 0
    if row['Loan Status'] == 'LIQUIDADO':
        return DPDUP if row['Moratorios'] > 0 else 0
    if pd.to_datetime(row['Application Date']) != pd.to_datetime(date.today()):
        return DPDUP if row['Moratorios'] > 0 else 0
    return DPDUP

payc['DPD']   = payc.apply(dpdp, axis=1)
payc['mvenc'] = payc.apply(lambda r: r['Capital'] if r['DPD'] > 0 else 0, axis=1)

print(f"payc: {payc.shape[0]:,} cuotas | {payc['Line'].nunique()} lineas | {payc['Person Id'].nunique()} clientes")

## 2. Líneas con DPD > 15 en alguna cuota

In [ ]:
# Monto total por línea — desde pay (una fila por ministración, sin expansión de cuotas)
monto_por_linea = pay.groupby('Line')['Amount'].sum().reset_index().rename(columns={'Amount': 'monto_total'})

# Fecha de vencimiento de la cuota con mayor DPD por línea
fecha_max_dpd = (
    payc.loc[payc.groupby('Line')['DPD'].idxmax(), ['Line', 'Fecha vencimiento cuota']]
    .rename(columns={'Fecha vencimiento cuota': 'fecha_venc_atraso'})
)

# Resumen por línea
resumen_lineas = (
    payc.groupby('Line').agg(
        Person_Id        = ('Person Id',   'first'),
        Nombre           = ('Name',        'first'),
        RFC              = ('RFC',         'first'),
        Originacion      = ('Originacion', 'first'),
        n_ministraciones = ('Portfolio Id','nunique'),
        dpd_max          = ('DPD',         'max'),
        cuotas_atraso    = ('DPD',         lambda x: (x > 0).sum()),
        total_cuotas     = ('DPD',         'count'),
        mvenc_total      = ('mvenc',       'sum'),
    ).reset_index()
)

resumen_lineas = (
    resumen_lineas
    .merge(monto_por_linea, on='Line', how='left')
    .merge(fecha_max_dpd,   on='Line', how='left')
)

resumen_lineas['pct_capital_vencido'] = (
    resumen_lineas['mvenc_total'] / resumen_lineas['monto_total'].replace(0, np.nan) * 100
).round(2)

# Status de la línea (ACTIVO / INACTIVO)
resumen_lineas['Status_Linea'] = resumen_lineas['Line'].map(
    data_ldc.set_index('Line')['Status']
)

# Filtrar a líneas con DPD > 15
atrasos = resumen_lineas[resumen_lineas['dpd_max'] > 15].copy()

print(f"Líneas con DPD > 15: {len(atrasos)} de {len(resumen_lineas)} ({len(atrasos)/len(resumen_lineas):.1%})")
print(f"  Activas:   {(atrasos['Status_Linea'] == 'ACTIVO').sum()}")
print(f"  Inactivas: {(atrasos['Status_Linea'] == 'INACTIVO').sum()}")

## 3. Detalle de líneas con atraso > 15 días

In [ ]:
cols_mostrar = [
    'Line', 'Nombre', 'RFC', 'Originacion', 'Status_Linea',
    'n_ministraciones', 'dpd_max', 'fecha_venc_atraso',
    'cuotas_atraso', 'total_cuotas', 'mvenc_total', 'monto_total', 'pct_capital_vencido'
]

detalle = atrasos[cols_mostrar].sort_values('dpd_max', ascending=False).reset_index(drop=True)

detalle['mvenc_total']       = detalle['mvenc_total'].map('${:,.0f}'.format)
detalle['monto_total']       = detalle['monto_total'].map('${:,.0f}'.format)
detalle['Originacion']       = pd.to_datetime(detalle['Originacion']).dt.strftime('%Y-%m')
detalle['fecha_venc_atraso'] = pd.to_datetime(detalle['fecha_venc_atraso']).dt.strftime('%Y-%m-%d')

print(detalle.to_string(index=False))

## 4. Resumen por status

In [ ]:
resumen_status = (
    atrasos.groupby('Status_Linea').agg(
        lineas          = ('Line',          'count'),
        clientes        = ('Person_Id',     'nunique'),
        dpd_max_promedio= ('dpd_max',       'mean'),
        dpd_max_total   = ('dpd_max',       'max'),
        capital_vencido = ('mvenc_total',   'sum'),
    ).reset_index()
)

resumen_status['dpd_max_promedio'] = resumen_status['dpd_max_promedio'].round(1)
resumen_status['capital_vencido']  = resumen_status['capital_vencido'].map('${:,.0f}'.format)

print(resumen_status.to_string(index=False))